# Waste Classification using VGG16
## Final Project - TensorFlow Transfer Learning

### Task 1: Import Libraries and Print TensorFlow Version

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models, optimizers

print("TensorFlow Version:", tf.__version__)

### Task 2: Define Dataset Directories and Create Generators

In [ ]:
train_dir = "dataset/train"
val_dir = "dataset/validation"
test_dir = "dataset/test"

train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

validation_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

### Task 3: Print Length of train_generator

In [ ]:
print("Length of train_generator:", len(train_generator))

### Task 4: Load VGG16 and Build Extract Features Model

In [ ]:
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

conv_base.trainable = False

extract_model = models.Sequential([
    conv_base,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

extract_model.summary()

### Task 5: Compile the Model

In [ ]:
extract_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

### Train Extract Features Model

In [ ]:
history = extract_model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator
)

### Task 6: Plot Accuracy Curves (Extract Features Model)

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy - Extract Features Model')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()

### Fine-Tuning Model

In [ ]:
conv_base.trainable = True

for layer in conv_base.layers[:15]:
    layer.trainable = False

fine_tune_model = extract_model

fine_tune_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

fine_tune_history = fine_tune_model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator
)

### Task 7: Plot Loss Curves (Fine Tune Model)

In [ ]:
plt.plot(fine_tune_history.history['loss'])
plt.plot(fine_tune_history.history['val_loss'])
plt.title('Loss - Fine Tune Model')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()

### Task 8: Plot Accuracy Curves (Fine Tune Model)

In [ ]:
plt.plot(fine_tune_history.history['accuracy'])
plt.plot(fine_tune_history.history['val_accuracy'])
plt.title('Accuracy - Fine Tune Model')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])
plt.show()

### Task 9: Plot Test Image using Extract Features Model (index_to_plot = 1)

In [ ]:
index_to_plot = 1
img_batch, label_batch = test_generator[index_to_plot]

prediction_extract = extract_model.predict(img_batch)

plt.imshow(img_batch[0])
plt.title("Extract Model Prediction: {:.2f}".format(prediction_extract[0][0]))
plt.axis('off')
plt.show()

### Task 10: Plot Test Image using Fine-Tuned Model (index_to_plot = 1)

In [ ]:
index_to_plot = 1
img_batch, label_batch = test_generator[index_to_plot]

prediction_fine = fine_tune_model.predict(img_batch)

plt.imshow(img_batch[0])
plt.title("Fine-Tuned Model Prediction: {:.2f}".format(prediction_fine[0][0]))
plt.axis('off')
plt.show()